# Tahap 3 — Forecast aliran air per zona gedung (interval 15 menit) dengan XGBoost
Lanjutan dari `xgboost_demand_forecast_colab.ipynb` (forecast per area distrik). Notebook ini melatih model
untuk **halaman Building Water System**: prediksi `flow_m3h` tiap zona (Z1–Z4) setiap 15 menit.

**Data:** tabel `usage_readings` dari `backend/data/water_twin.db` (atau CSV format import dashboard:
`ts` (UTC), `zone_id`, `flow_m3h` atau `volume_m3`, `source`).

**Target pembanding:** metode yang dipakai dashboard sekarang (`usage.DemandProvider`) =
rata-rata jam-dalam-minggu 8 minggu terakhir. XGBoost baru layak dipasang kalau mengalahkan ini di data test.

> ⚠ Data bawaan demo berlabel **SIMULATED**: dibangkitkan dari kurva harian tetap + noise. Untuk data seperti itu,
> rata-rata jam-dalam-minggu sudah hampir optimal, jadi selisih XGBoost vs baseline kemungkinan kecil.
> Hasil yang bermakna butuh data meter asli (`MEASURED`).

Jalankan dari atas ke bawah. Bisa di **Google Colab** (upload file) atau **Jupyter lokal** di folder `notebooks/`
(otomatis membaca `../backend/data/water_twin.db`).

## Langkah 0 — Setup

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Samakan versi xgboost dengan backend (requirements: xgboost>=3.1, venv backend: 3.4.1)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna", "xgboost==3.4.1"], check=True)

import json, sqlite3, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import optuna
from sklearn.model_selection import TimeSeriesSplit
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 160)
print("xgboost", xgb.__version__, "| optuna", optuna.__version__, "| Colab" if IN_COLAB else "| lokal")

## Langkah 1 — Konfigurasi
Nilai default mengikuti `backend/data/building.json`.

In [ ]:
TZ_OFFSET_H = 3       # building.json -> timezone_offset_h (data di DB disimpan dalam UTC)
STEP_MIN    = 15      # interval data (building.json -> synthetic.interval_min)
TEST_DAYS   = 7       # hari terakhir sebagai test (sama dengan back-test di tab Usage)
N_TRIALS    = 30      # Optuna; set 5 untuk uji cepat
SEED        = 42

STEPS_PER_DAY = 24 * 60 // STEP_MIN    # 96 slot per hari
STEP = pd.Timedelta(minutes=STEP_MIN)
FREQ = f"{STEP_MIN}min"
OUT_DIR = Path("/content") if IN_COLAB else Path("outputs_building_flow")
OUT_DIR.mkdir(exist_ok=True)
print("Output ->", OUT_DIR.resolve())

## Langkah 2 — Load data
`DATA_FILE = None` artinya: di lokal pakai `../backend/data/water_twin.db`; di Colab muncul dialog upload
(upload `water_twin.db` atau file CSV).

In [ ]:
DATA_FILE = None   # contoh: "/content/water_twin.db" atau "../backend/data/water_twin.db" atau "usage.csv"

if DATA_FILE is None:
    local_db = Path("../backend/data/water_twin.db")
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        DATA_FILE = "/content/" + list(up.keys())[0]
    elif local_db.exists():
        DATA_FILE = str(local_db)
    else:
        raise FileNotFoundError("Set DATA_FILE ke water_twin.db atau CSV usage")

if DATA_FILE.endswith((".db", ".sqlite", ".sqlite3")):
    with sqlite3.connect(DATA_FILE) as con:
        raw = pd.read_sql_query("SELECT ts, zone_id, flow_m3h, source FROM usage_readings", con)
else:
    raw = pd.read_csv(DATA_FILE)
    raw.columns = [c.strip().lower() for c in raw.columns]
    if "source" not in raw:
        raw["source"] = "CSV"

raw["ts"] = pd.to_datetime(raw["ts"], utc=True).dt.tz_localize(None)     # UTC naif, sama seperti DB
if "flow_m3h" not in raw:
    # Sama seperti backend (db.values_to_flows): volume per interval -> m3/jam
    raw = raw.sort_values(["zone_id", "ts"])
    dt_h = raw.groupby("zone_id")["ts"].diff().dt.total_seconds() / 3600
    dt_h = dt_h.fillna(dt_h.groupby(raw["zone_id"]).transform("median"))
    raw["flow_m3h"] = raw["volume_m3"] / dt_h
raw = raw.dropna(subset=["ts", "zone_id", "flow_m3h"])
raw["zone_id"] = raw["zone_id"].astype(str)
raw["local"] = raw["ts"] + pd.Timedelta(hours=TZ_OFFSET_H)               # semua fitur waktu pakai jam lokal

print("File:", DATA_FILE)
display(raw.groupby(["source", "zone_id"]).agg(rows=("flow_m3h", "size"), first_local=("local", "min"),
                                                last_local=("local", "max"), mean_m3h=("flow_m3h", "mean")))
SOURCES = sorted(raw["source"].unique())
if SOURCES == ["SIMULATED"]:
    print("⚠ Semua baris SIMULATED: hasil ini menguji pipeline, bukan akurasi di gedung nyata.")

## Langkah 3 — Grid 15 menit yang rapat per zona
Lag dihitung dengan `shift(n)` per zona, jadi setiap zona harus punya satu baris per slot 15 menit tanpa loncatan.
Slot yang tidak ada bacaannya diisi `NaN` (tidak dipakai sebagai target; XGBoost membaca NaN di fitur sebagai *missing*).
Timestamp = **awal interval**, waktu lokal.

In [ ]:
ZONES = sorted(raw["zone_id"].unique())
frames = []
for z in ZONES:
    s = raw[raw.zone_id == z].assign(local=lambda d: d["local"].dt.floor(FREQ)).groupby("local")["flow_m3h"].mean()
    grid = pd.date_range(s.index.min(), s.index.max(), freq=FREQ)
    s = s.reindex(grid)
    frames.append(pd.DataFrame({"local": grid, "zone_id": z, "flow_m3h": s.to_numpy()}))
    print(f"{z}: {len(grid):>6,} slot | {int(s.isna().sum())} slot kosong")
data = pd.concat(frames, ignore_index=True)

## Langkah 4 — Fitur (semua lag ≥ 1 hari, anti-bocor)
Di dashboard, prediksi dipakai sampai **24 jam ke depan** (grafik "Next 24 h forecast") dan **2 jam ke depan**
(load balancer mode prediktif). Karena semua lag **≥ 96 slot (1 hari)**, prediksi untuk jam mana pun dalam 24 jam
ke depan hanya memakai data yang sudah tersedia saat ini. Untuk lebih dari 24 jam, prediksi dipakai sebagai lag
berikutnya (rekursif, diuji di Langkah 10).

| Fitur | Arti |
|---|---|
| `zone_id` | zona (kategori) |
| `slot`, `hour`, `day_of_week`, `is_weekend`, sin/cos | posisi dalam hari dan minggu |
| `lag_1d`, `lag_2d`, `lag_7d` | aliran pada slot yang sama 1, 2, 7 hari lalu |
| `slot_mean_7d` | rata-rata slot yang sama selama 7 hari terakhir |
| `roll_mean_1d`, `roll_std_1d`, `roll_mean_7d` | level dan variasi aliran (deret yang sudah di-shift 1 hari) |

Seperti notebook sebelumnya, target dan lag dibagi **rata-rata zona dari data train** (`zone_mean`), sehingga satu
model global bisa belajar dari zona dengan skala berbeda. Fungsi `add_features` ini yang nanti dipakai juga di backend.

In [ ]:
D = STEPS_PER_DAY
LAG_STEPS = {"lag_1d": D, "lag_2d": 2 * D, "lag_7d": 7 * D}
ROLLS = ["roll_mean_1d", "roll_std_1d", "roll_mean_7d", "slot_mean_7d"]
LAGS = list(LAG_STEPS) + ROLLS
CALENDAR = ["slot", "hour", "day_of_week", "is_weekend", "sin_slot", "cos_slot", "sin_dow", "cos_dow"]
FEATURES = ["zone_id"] + CALENDAR + [c + "_n" for c in LAGS]


def add_features(df, zone_mean):
    # df: kolom local, zone_id, flow_m3h di grid 15 menit yang rapat per zona (NaN = belum diketahui).
    df = df[["local", "zone_id", "flow_m3h"]].copy()
    df["zone_id"] = df["zone_id"].astype(str)
    df = df.sort_values(["zone_id", "local"]).reset_index(drop=True)
    t = df["local"]
    df["slot"] = t.dt.hour * (60 // STEP_MIN) + t.dt.minute // STEP_MIN
    df["hour"] = t.dt.hour
    df["day_of_week"] = t.dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["sin_slot"] = np.sin(2 * np.pi * df["slot"] / D)
    df["cos_slot"] = np.cos(2 * np.pi * df["slot"] / D)
    df["sin_dow"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["cos_dow"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    g = df.groupby("zone_id")["flow_m3h"]
    for name, k in LAG_STEPS.items():
        df[name] = g.shift(k)
    df["slot_mean_7d"] = pd.concat([g.shift(k * D) for k in range(1, 8)], axis=1).mean(axis=1)
    gs = g.shift(D).groupby(df["zone_id"])
    df["roll_mean_1d"] = gs.transform(lambda s: s.rolling(D, min_periods=D // 2).mean())
    df["roll_std_1d"] = gs.transform(lambda s: s.rolling(D, min_periods=D // 2).std())
    df["roll_mean_7d"] = gs.transform(lambda s: s.rolling(7 * D, min_periods=7 * D // 2).mean())

    df["zone_mean"] = df["zone_id"].map(zone_mean).astype(float)
    for c in LAGS:
        df[c + "_n"] = df[c] / df["zone_mean"]
    df["zone_id"] = pd.Categorical(df["zone_id"], categories=ZONES)
    return df


# Split berdasarkan waktu: TEST_DAYS hari terakhir (mulai tengah malam lokal)
SPLIT = (data["local"].max() + STEP).floor("D") - pd.Timedelta(days=TEST_DAYS)
zone_mean = data[data["local"] < SPLIT].groupby("zone_id")["flow_m3h"].mean()

df = add_features(data, zone_mean)
df["y"] = df["flow_m3h"] / df["zone_mean"]
df = df.dropna(subset=["y", "lag_7d"]).reset_index(drop=True)   # minggu pertama belum punya lag 7 hari
print(len(FEATURES), "fitur | split lokal:", SPLIT, "| baris:", len(df))
display(zone_mean.rename("zone_mean_m3h").to_frame())

## Langkah 5 — Split train / test (berdasarkan waktu, tidak diacak)

In [ ]:
train = df[df["local"] < SPLIT].reset_index(drop=True)
test = df[df["local"] >= SPLIT].reset_index(drop=True)
X_tr, y_tr = train[FEATURES], train["y"]
X_te = test[FEATURES]
print("train", X_tr.shape, train.local.min(), "->", train.local.max())
print("test ", X_te.shape, test.local.min(), "->", test.local.max())

## Langkah 6 — Baseline: metode dashboard sekarang
Implementasi ulang `backend/app/usage.py::DemandProvider`: rata-rata per jam-dalam-minggu (168 slot) dari 8 minggu
terakhir data **sebelum** split, diinterpolasi linear antar titik tengah jam, dievaluasi di titik tengah interval.
Baseline tidak butuh lag, jadi memakai semua data sebelum split (termasuk minggu pertama).
Ditambah dua baseline naif: slot yang sama **kemarin** dan **minggu lalu**.

In [ ]:
def dashboard_profile(hist):
    h = hist.dropna(subset=["flow_m3h"]).assign(hour_start=lambda d: d["local"].dt.floor("h"))
    h = h.groupby(["zone_id", "hour_start"], as_index=False)["flow_m3h"].mean()
    h = h[h["hour_start"] > h["hour_start"].max() - pd.Timedelta(weeks=8)]
    h["how"] = h["hour_start"].dt.dayofweek * 24 + h["hour_start"].dt.hour
    h["hod"] = h["hour_start"].dt.hour
    prof = {}
    for z, g in h.groupby("zone_id"):
        st = g.groupby("how")["flow_m3h"].agg(["mean", "count"]).reindex(range(168))
        hod = g.groupby("hod")["flow_m3h"].mean().reindex(range(24)).fillna(0.0)
        sparse = st["count"].isna() | (st["count"] < 3)
        prof[z] = np.where(sparse, hod.to_numpy()[np.arange(168) % 24], st["mean"].to_numpy())
    return prof


def dashboard_predict(prof, zone_ids, local):
    pos = pd.Series(local) + STEP / 2 - pd.Timedelta(minutes=30)
    h0 = (pos.dt.dayofweek * 24 + pos.dt.hour).to_numpy()
    w = ((pos.dt.minute * 60 + pos.dt.second) / 3600).to_numpy()
    out = np.full(len(pos), np.nan)
    zone_ids = np.asarray(zone_ids, str)
    for z, mu in prof.items():
        m = zone_ids == z
        out[m] = (1 - w[m]) * mu[h0[m]] + w[m] * mu[(h0[m] + 1) % 168]
    return out


prof = dashboard_profile(data[data["local"] < SPLIT])
test["baseline"] = dashboard_predict(prof, test["zone_id"], test["local"])
test["naive_1d"] = test["lag_1d"]
test["naive_7d"] = test["lag_7d"]


def wape(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = ~(np.isnan(a) | np.isnan(p))
    return np.abs(a[m] - p[m]).sum() / np.abs(a[m]).sum()


def mae(a, p):
    a, p = np.asarray(a, float), np.asarray(p, float)
    m = ~(np.isnan(a) | np.isnan(p))
    return np.abs(a[m] - p[m]).mean()


print(f"WAPE baseline dashboard: {wape(test.flow_m3h, test.baseline):.2%} | "
      f"kemarin: {wape(test.flow_m3h, test.naive_1d):.2%} | minggu lalu: {wape(test.flow_m3h, test.naive_7d):.2%}")

## Langkah 7 — Validasi expanding window + tuning Optuna
Fold dibagi berdasarkan timestamp unik (semua zona di jam yang sama masuk fold yang sama).
WAPE dihitung dalam m³/jam (prediksi × `zone_mean`), bukan pada target ternormalisasi.

In [ ]:
uniq_ts = np.sort(train["local"].unique())
FOLDS = []
for tr_i, va_i in TimeSeriesSplit(n_splits=5).split(uniq_ts):
    FOLDS.append((np.where(train["local"].isin(uniq_ts[tr_i]))[0], np.where(train["local"].isin(uniq_ts[va_i]))[0]))
for k, (a, b) in enumerate(FOLDS):
    print(f"fold {k}: train {len(a):>6,} | valid {len(b):>5,} | valid mulai {train.local.iloc[b[0]]}")

BASE = dict(tree_method="hist", enable_categorical=True, random_state=SEED, n_jobs=-1)


def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 30, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    )
    scores, iters = [], []
    for tr_i, va_i in FOLDS[-3:]:
        m = xgb.XGBRegressor(**params, **BASE, n_estimators=2000, early_stopping_rounds=50)
        m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i], eval_set=[(X_tr.iloc[va_i], y_tr.iloc[va_i])], verbose=False)
        p = m.predict(X_tr.iloc[va_i]) * train["zone_mean"].iloc[va_i].to_numpy()
        scores.append(wape(train["flow_m3h"].iloc[va_i], p))
        iters.append(m.best_iteration)
    trial.set_user_attr("n_estimators", int(np.mean(iters)) + 1)
    return float(np.mean(scores))


t0 = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=not IN_COLAB)
best_params = study.best_params
best_n = study.best_trial.user_attrs["n_estimators"]
print(f"selesai dalam {time.time() - t0:.0f}s | CV WAPE terbaik: {study.best_value:.4f}")
print("Parameter:", best_params, "| n_estimators:", best_n)

## Langkah 8 — Pita ketidakpastian (σ) dari prediksi out-of-fold
Dashboard menampilkan rentang **mean ± σ** dan memakai `cv` (σ/mean) per zona untuk uji keamanan load balancer
("high demand +cv"). σ di sini diambil dari residual out-of-fold (3 fold terakhir), **bukan** dari data test,
supaya cakupan ±1σ di test bisa dinilai jujur. Nilai `cv` di-clip ke `[0.08, 0.30]` seperti `building.json -> forecast`.

In [ ]:
CV_MIN, CV_MAX = 0.08, 0.30
oof = []
for tr_i, va_i in FOLDS[-3:]:
    m = xgb.XGBRegressor(**best_params, **BASE, n_estimators=best_n)
    m.fit(X_tr.iloc[tr_i], y_tr.iloc[tr_i], verbose=False)
    v = train.iloc[va_i]
    oof.append(v[["zone_id", "hour", "flow_m3h"]].assign(pred=m.predict(X_tr.iloc[va_i]) * v["zone_mean"].to_numpy()))
oof = pd.concat(oof)
oof["res"] = oof["flow_m3h"] - oof["pred"]

sigma = (oof.groupby(["zone_id", "hour"], observed=True)["res"].std()
            .unstack("hour").reindex(index=ZONES, columns=range(24)))
sigma = sigma.T.fillna(sigma.mean(axis=1)).T.fillna(0.0)            # jam tanpa data -> rata-rata zona
cv_zone = {}
for z in ZONES:
    o = oof[oof.zone_id == z]
    ok = o.groupby("hour")["flow_m3h"].mean()
    ratio = (sigma.loc[z] / ok.reindex(range(24))).where(ok.reindex(range(24)) > 0.5).median()
    cv_zone[z] = float(np.clip(CV_MIN if np.isnan(ratio) else ratio, CV_MIN, CV_MAX))
print("cv per zona:", {z: round(v, 3) for z, v in cv_zone.items()})
display(sigma.round(2))

## Langkah 9 — Latih model akhir pada seluruh train

In [ ]:
cut = train["local"].quantile(0.9)              # 10% waktu terakhir train untuk early stopping
fit_m, val_m = train["local"] < cut, train["local"] >= cut
model = xgb.XGBRegressor(**best_params, **BASE, n_estimators=max(best_n * 2, 300), early_stopping_rounds=50)
model.fit(X_tr[fit_m], y_tr[fit_m], eval_set=[(X_tr[fit_m], y_tr[fit_m]), (X_tr[val_m], y_tr[val_m])], verbose=200)
print("best_iteration:", model.best_iteration)

hist = model.evals_result()
plt.figure(figsize=(7, 3))
plt.plot(hist["validation_0"]["rmse"], label="train")
plt.plot(hist["validation_1"]["rmse"], label="valid")
plt.xlabel("iterasi"); plt.ylabel("RMSE (ternormalisasi)"); plt.legend(); plt.title("Kurva belajar")
plt.show()

## Langkah 10 — Evaluasi di test
Dua cara memakai XGBoost dinilai:
1. **Day-ahead**: lag memakai data aktual (prediksi dibuat ≤ 24 jam sebelumnya). Ini kondisi grafik 24 jam dan
   load balancer 2 jam.
2. **Rekursif**: hanya data sebelum split yang diketahui; seluruh periode test diprediksi hari demi hari, dan
   prediksi hari sebelumnya dipakai sebagai lag. Ini kondisi twin yang jam simulasinya berjalan melewati data terakhir.

WAPE juga dihitung pada rata-rata per jam supaya bisa dibandingkan dengan back-test di tab Usage (yang per jam).

In [ ]:
def predict_m3h(rows):
    return np.clip(model.predict(rows[FEATURES]) * rows["zone_mean"].to_numpy(), 0, None)


def forecast_recursive(hist, start, n_days):
    # Prediksi n_days mulai `start` hanya dari baris sebelum `start`; hasil tiap hari jadi lag hari berikutnya.
    known = hist.loc[(hist["local"] < start) & (hist["local"] >= start - pd.Timedelta(days=9)),
                     ["local", "zone_id", "flow_m3h"]]
    out = []
    for d in range(n_days):
        t0 = start + pd.Timedelta(days=d)
        times = pd.date_range(t0, periods=D, freq=FREQ)
        block = pd.DataFrame({"local": np.tile(times, len(ZONES)), "zone_id": np.repeat(ZONES, D), "flow_m3h": np.nan})
        f = add_features(pd.concat([known, block], ignore_index=True), zone_mean)
        rows = f[f["local"] >= t0]
        block = pd.DataFrame({"local": rows["local"].to_numpy(), "zone_id": rows["zone_id"].astype(str).to_numpy(),
                              "flow_m3h": predict_m3h(rows)})
        known = pd.concat([known, block], ignore_index=True)
        out.append(block)
    return pd.concat(out, ignore_index=True)


test["xgb"] = predict_m3h(test)
n_days = int(np.ceil((test["local"].max() + STEP - SPLIT) / pd.Timedelta(days=1)))
rec = forecast_recursive(data, SPLIT, n_days).rename(columns={"flow_m3h": "xgb_recursive"})
test = test.drop(columns="xgb_recursive", errors="ignore").merge(
    rec.assign(zone_id=lambda d: pd.Categorical(d.zone_id, categories=ZONES)), on=["local", "zone_id"], how="left")
test["sigma"] = [sigma.loc[z, h] for z, h in zip(test["zone_id"].astype(str), test["hour"])]

hourly_t = (test.assign(hour_start=test["local"].dt.floor("h"))
                .groupby(["zone_id", "hour_start"], observed=True)[["flow_m3h", "xgb", "xgb_recursive", "baseline",
                                                                    "naive_1d", "naive_7d"]].mean())
METHODS = {"xgb": "XGBoost (day-ahead)", "xgb_recursive": "XGBoost (rekursif)",
           "baseline": "Baseline dashboard (jam-dalam-minggu)", "naive_1d": "Naif: kemarin", "naive_7d": "Naif: minggu lalu"}
results = pd.DataFrame({
    "MAE (m3/jam)": [mae(test.flow_m3h, test[c]) for c in METHODS],
    "WAPE 15 menit": [wape(test.flow_m3h, test[c]) for c in METHODS],
    "WAPE per jam": [wape(hourly_t.flow_m3h, hourly_t[c]) for c in METHODS],
}, index=list(METHODS.values()))
display(results.style.format({"MAE (m3/jam)": "{:.3f}", "WAPE 15 menit": "{:.2%}", "WAPE per jam": "{:.2%}"}))

within = float(np.mean(np.abs(test.flow_m3h - test.xgb) <= test.sigma))
gain = 1 - results.loc[METHODS["xgb"], "WAPE 15 menit"] / results.loc[METHODS["baseline"], "WAPE 15 menit"]
print(f"Cakupan ±1σ (day-ahead): {within:.1%}  (normal ≈ 68%)")
print(f"XGBoost vs baseline dashboard: WAPE {'turun' if gain > 0 else 'naik'} {abs(gain):.1%}")

per_zone = pd.DataFrame({METHODS[c]: test.groupby("zone_id", observed=True).apply(lambda d: wape(d.flow_m3h, d[c]))
                         for c in ["xgb", "xgb_recursive", "baseline"]})
display(per_zone.style.format("{:.2%}"))

In [ ]:
fig, axes = plt.subplots(len(ZONES), 1, figsize=(14, 2.6 * len(ZONES)), sharex=True)
w = test[test["local"] < SPLIT + pd.Timedelta(days=3)]
for ax, z in zip(np.atleast_1d(axes), ZONES):
    d = w[w.zone_id == z]
    ax.plot(d.local, d.flow_m3h, label="aktual", color="black", lw=1)
    ax.plot(d.local, d.xgb, label="XGBoost day-ahead", color="#0072bc")
    ax.fill_between(d.local, (d.xgb - d.sigma).clip(lower=0), d.xgb + d.sigma, color="#0072bc", alpha=.15, label="±1σ")
    ax.plot(d.local, d.baseline, label="baseline dashboard", color="#a65800", ls="--")
    ax.set_ylabel(f"{z} m³/jam")
np.atleast_1d(axes)[0].legend(ncol=4, loc="upper left"); plt.suptitle("3 hari pertama test"); plt.tight_layout(); plt.show()

by_h = test.groupby("hour").apply(lambda d: pd.Series({"XGBoost": wape(d.flow_m3h, d.xgb),
                                                       "Baseline": wape(d.flow_m3h, d.baseline)}))
by_h.plot.bar(figsize=(12, 3), title="WAPE per jam (test)", color=["#0072bc", "#a65800"]); plt.show()

## Langkah 11 — Feature importance

In [ ]:
imp = pd.Series(model.get_booster().get_score(importance_type="gain")).sort_values()
imp.plot.barh(figsize=(6, 5), title="Feature importance (gain)"); plt.show()

## Langkah 12 — Simpan model & hasil
File yang dihasilkan (kirim semuanya untuk integrasi ke dashboard):

| File | Isi |
|---|---|
| `xgb_building_flow.json` | model XGBoost |
| `xgb_building_flow_meta.json` | fitur, kategori zona, `zone_mean`, lag (dalam slot), σ per zona × jam, `cv`, metrik test, parameter |
| `building_flow_test_predictions.csv` | prediksi test per slot: aktual, XGBoost (day-ahead dan rekursif), baseline, σ |

In [ ]:
model.save_model(str(OUT_DIR / "xgb_building_flow.json"))
meta = {
    "model_file": "xgb_building_flow.json",
    "xgboost_version": xgb.__version__,
    "target": "flow_m3h / zone_mean",
    "step_min": STEP_MIN,
    "tz_offset_h": TZ_OFFSET_H,
    "timestamp": "local time (UTC + tz_offset_h) at the START of each interval",
    "features": FEATURES,
    "categories": {"zone_id": ZONES},
    "lag_steps": LAG_STEPS,
    "rolls": {"roll_mean_1d": "mean of last 1 day, shifted 1 day", "roll_std_1d": "std of last 1 day, shifted 1 day",
              "roll_mean_7d": "mean of last 7 days, shifted 1 day", "slot_mean_7d": "mean of same slot, 1..7 days ago"},
    "min_lag_steps": min(LAG_STEPS.values()),
    "zone_mean": {z: float(v) for z, v in zone_mean.items()},
    "sigma_m3h_by_hour": {z: [float(x) for x in sigma.loc[z]] for z in ZONES},
    "cv": cv_zone,
    "split_local": str(SPLIT),
    "train_range_local": [str(train.local.min()), str(train.local.max())],
    "test_range_local": [str(test.local.min()), str(test.local.max())],
    "data_sources": SOURCES,
    "rows": {"train": int(len(train)), "test": int(len(test))},
    "params": best_params,
    "best_iteration": int(model.best_iteration),
    "cv_wape": float(study.best_value),
    "metrics_test": {name: {k: float(v) for k, v in row.items()} for name, row in results.iterrows()},
    "within_1sigma_test": within,
}
(OUT_DIR / "xgb_building_flow_meta.json").write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
cols = ["local", "zone_id", "flow_m3h", "xgb", "xgb_recursive", "baseline", "naive_1d", "naive_7d", "sigma"]
test[cols].to_csv(OUT_DIR / "building_flow_test_predictions.csv", index=False)
print("Tersimpan di", OUT_DIR.resolve())

if IN_COLAB:
    from google.colab import files
    for f in ["xgb_building_flow.json", "xgb_building_flow_meta.json", "building_flow_test_predictions.csv"]:
        files.download(str(OUT_DIR / f))

# Memuat ulang nanti:
# b = xgb.Booster(); b.load_model("xgb_building_flow.json")
# pred = b.predict(xgb.DMatrix(rows[FEATURES], enable_categorical=True),
#                  iteration_range=(0, int(b.attr("best_iteration")) + 1)) * rows["zone_mean"]